# Imports

In [10]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Preprocessing

In [3]:
df = pd.read_csv('no_sales_df.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,Order Date,Ship Date,Ship Mode,Segment,City,State,Country,Market,...,Discount,Profit,Shipping Cost,Order Priority,Price,Avg_Sales_Category,Avg_Sales_Country,Avg_Sales_Market,Avg_Sales_Region,Discount_value
0,0,234,2012-03-30,2012-04-01,First Class,Consumer,Cairo,Al Qahirah,Egypt,Africa,...,0.0,140.1600,399.96,Critical,637.35,467.858939,172.770678,170.868370,170.868370,0.00
1,1,238,2014-12-23,2014-12-26,First Class,Consumer,Detroit,Michigan,United States,US,...,0.0,412.5394,397.52,High,226.67,416.248905,229.858001,229.858001,253.872674,0.00
2,2,239,2014-07-04,2014-07-04,Same Day,Home Office,Seattle,Washington,United States,US,...,0.2,209.5800,396.92,High,399.20,467.858939,229.858001,229.858001,226.493233,479.04
3,3,240,2014-11-21,2014-11-23,First Class,Consumer,New York City,New York,United States,US,...,0.0,327.5922,394.57,Critical,419.99,467.858939,229.858001,229.858001,238.336110,0.00
4,4,241,2013-01-23,2013-01-27,Standard Class,Consumer,Baku,Baki,Azerbaijan,EMEA,...,0.0,946.6800,393.62,High,514.50,416.248905,194.190000,160.302508,160.302508,0.00


In [4]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

In [5]:
X = df.drop(['Price', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1)
y = df['Price']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

# GridSearch

In [7]:
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'iterations': [500, 1000, 1500]
}

model = CatBoostRegressor(
    loss_function='RMSE',
    random_state=42,
    cat_features=categorical_features,
    verbose=0
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
 

# Training

In [8]:
best_model = CatBoostRegressor(
    **best_params,
    loss_function='RMSE',
    random_state=42,
    cat_features=categorical_features,
    verbose=100
)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
val_pool = Pool(X_val, y_val, cat_features=categorical_features)

best_model.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=50,
    use_best_model=True,
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 90.1994911	test: 91.0918257	best: 91.0918257 (0)	total: 17.1ms	remaining: 25.6s
100:	learn: 28.0096471	test: 29.9716408	best: 29.9716408 (100)	total: 1.68s	remaining: 23.3s
200:	learn: 24.4947548	test: 27.3447516	best: 27.3447516 (200)	total: 3.32s	remaining: 21.4s
300:	learn: 22.7651323	test: 26.3785211	best: 26.3765778 (299)	total: 4.89s	remaining: 19.5s
400:	learn: 21.7656583	test: 25.9898276	best: 25.9898276 (400)	total: 6.56s	remaining: 18s
500:	learn: 20.9083149	test: 25.5868813	best: 25.5860486 (499)	total: 8.18s	remaining: 16.3s
600:	learn: 20.3086883	test: 25.3876230	best: 25.3857594 (599)	total: 9.85s	remaining: 14.7s
700:	learn: 19.7286580	test: 25.2142952	best: 25.2131078 (697)	total: 11.5s	remaining: 13.1s
800:	learn: 19.2007296	test: 25.1341940	best: 25.1341940 (800)	total: 13.3s	remaining: 11.6s
900:	learn: 18.7372730	test: 25.0552581	best: 25.0552581 (900)	total: 15.4s	remaining: 10.2s
1000:	learn: 18.3146383	test: 24.9883850	best: 24.9875333 (997)	total: 17.9

In [11]:
y_test_pred = best_model.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_test_pred)
mse_ridge = mean_squared_error(y_test, y_test_pred)
rmse_ridge = mean_squared_error(y_test, y_test_pred, squared=False)
r2_ridge = r2_score(y_test, y_test_pred)

print('Metrics:')
print(f"MAE: {mae_ridge}")
print(f"MSE: {mse_ridge}")
print(f"RMSE: {rmse_ridge}")
print(f"R² Score: {r2_ridge}")

Metrics:
MAE: 11.5403661700564
MSE: 662.4199493990728
RMSE: 25.73752026515128
R² Score: 0.9305460302789295


/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
